In [ ]:
import pinns
import numpy as np
import pickle
import json
import os
from datetime import datetime

In [ ]:

pinns.use_backend("jax")

# ---------------- Physics parameters (paper) ----------------
lam = 1.0
eps = 0.05

# ---------------- Domain (paper) ----------------
domain = pinns.DomainCubic(
    xmin=[-1.0, -1.0, 0.0],
    xmax=[ 1.0,  1.0, 1.0]
)

# ---------------- Initial condition (bubble merging) ----------------
def initial_condition(X):
    x = X[:,0:1]
    y = X[:,1:2]

    r0 = 0.4
    R1 = np.sqrt((x - 0.7*r0)**2 + y**2)
    R2 = np.sqrt((x + 0.7*r0)**2 + y**2)

    phi1 = np.tanh((r0 - R1) / (2*eps))
    phi2 = np.tanh((r0 - R2) / (2*eps))

    return np.maximum(phi1, phi2)

domain.add_dirichlet(
    boundary=(None, None, 0),
    value=initial_condition,
    component=0,
    name="initial"
)

# ---------------- Periodic BCs in x,y ----------------
domain.add_periodic(dim=0, name="periodic_x", component=0, match_x_derivative=True)
domain.add_periodic(dim=1, name="periodic_y", component=0, match_x_derivative=True)

In [ ]:

# ---------------- Cahn–Hilliard PDE ----------------
# φ_t = λ Δ μ
# μ = -ε² Δφ + φ³ - φ
def cahn_hilliard_2d(X, V, params, derivative=None):
    if derivative is None:
        derivative = pinns.derivative

    lam_val = params["fixed"]["lam"]
    eps_val = params["fixed"]["eps"]

    phi = V[:,0]   # (N,)
    mu  = V[:,1]   # (N,)

    # Time derivative
    phi_t = derivative(V, X, component=0, order=(2,))

    # Laplacians
    phi_xx = derivative(V, X, component=0, order=(0,0))
    phi_yy = derivative(V, X, component=0, order=(1,1))
    lap_phi = phi_xx + phi_yy

    mu_xx = derivative(V, X, component=1, order=(0,0))
    mu_yy = derivative(V, X, component=1, order=(1,1))
    lap_mu = mu_xx + mu_yy

    # PDE system
    eq1 = phi_t - lam_val * lap_mu
    eq2 = mu - (-eps_val**2 * lap_phi + phi**3 - phi)

    return [eq1, eq2]

def dummy_solution(X, params):
    return np.zeros((X.shape[0], 2))

In [ ]:
# ---------------- Problem ----------------
problem = pinns.Problem(
    domain=domain,
    pde_fn=cahn_hilliard_2d,
    input_names=["x","y","t"],
    output_names=["phi","mu"],
    output_range=(-1,1),
    params={"lam": lam, "eps": eps},
    # lagrange_multipliers=["pde", "initial", "periodic_x", "periodic_y"]
    solution=dummy_solution
)

# ---------------- Network ----------------
network = pinns.FNN(
    layer_sizes=[3, 64, 64, 64, 2],
    activation="tanh",
    normalize_input=True,
    unnormalize_output=True
)

trainer = pinns.Trainer(problem, network)

# ---------------- Stage 1: Adam ----------------
trainer.compile(
    train_samples={
        "pde": 10000,
        "initial": 1000
    },
    test_samples={
        "pde": 500,
        "initial": 100
    },
    weights={
        "pde": 1.0,
        "initial": 1.0,
        "periodic_x": 1.0,
        "periodic_y": 1.0
    },
    optimizer="adam",
    learning_rate=1e-3,
    epochs=1000,
    print_each=500,
    show_plots=True,
    adaptive_sampling=False,
    adaptive_each=1000,
    adaptive_ratio=0.5,
    adaptive_std=0.1,
    adaptive_mode="replace",
    lagrange_lr=0.0
)

trainer.train()

# # ---------------- Stage 2: L-BFGS ----------------
# trainer.compile(
#     optimizer="lbfgs",
#     epochs=500,
#     print_each=50,
#     show_plots=True
# )

# trainer.train()

In [ ]:

# ---------------- Save checkpoint ----------------
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
save_dir = f"{timestamp}_CahnHilliard2D"
os.makedirs(save_dir, exist_ok=True)

with open(os.path.join(save_dir, "last_checkpoint.pkl"), "wb") as f:
    pickle.dump(trainer.network.params, f)

manifest = {
    "model": {
        "layer_sizes": network.layer_sizes,
        "activation": network.activation,
        "normalize_input": network.normalize_input,
        "unnormalize_output": network.unnormalize_output
    },
    "physics": {
        "pde": "cahn_hilliard_2d",
        "lam": lam,
        "eps": eps,
        "domain": {"x":[-1,1], "y":[-1,1], "t":[0,1]}
    },
    "files": {"params": "last_checkpoint.pkl"}
}

with open(os.path.join(save_dir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=4)


with open(os.path.join(save_dir, "loss_history.pkl"), "wb") as f:
    pickle.dump(trainer.get_history(), f)


print("Saved:", save_dir)
